# Variantes de réglage du CNN-LSTM (une modification à la fois)

Le protocole et le critère sont décrits dans **NOTES.md §6**, écrits avant les essais.

- **Critère unique :** la perte de validation retenue, moyennée sur les 5 plis (sujets de validation pris dans le train). **Jamais le test.**
- **Référence « sans réglage » :** les dossiers `cd_cnn_4cl`, `cnn_lstm32_4cl`, `cd_cnn_2cl` et `cnn_lstm32_2cl`, produits par `colab_run.ipynb`. Ce notebook ne les modifie pas : chaque variante écrit dans son propre dossier.
- **Criblage en 4 classes avec `--sans-adaptation` :** seul l'entraînement global est lancé, car le critère n'en dépend pas.
- **Sauvegarde :** après chaque pli, sur Drive (`OUT_DIR`). Après une déconnexion, relancer les cellules d'installation, puis la cellule de la variante en cours : les plis terminés sont relus.

**Ordre :**
1. Installation, puis vérification de la référence et estimation des temps.
2. Variantes (a), (b), (c) et (d) en 4 classes : **une cellule par variante**, à lancer une par une.
3. Comparaison en 4 classes.
4. Confirmation en 2 classes de la variante retenue, et ajout de l'adaptation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paramètres (les mêmes que `colab_run.ipynb`)

In [ ]:
ZIP_DRIVE = '/content/drive/MyDrive/eegmmidb-1.0.0.zip'
DATA_DIR  = '/content/eegmmidb-1.0.0/files'
OUT_DIR   = '/content/drive/MyDrive/eeg_resultats'          # même dossier que la référence
CACHE_DIR = '/content/cache'
REPO_URL  = 'https://github.com/ChaYLbd123/EEG_Classification.git'
REPO_DIR  = '/content/EEG_Classification'

In [ ]:
import os
if not os.path.isdir(DATA_DIR):
    !unzip -q -n "$ZIP_DRIVE" -d /content/eegmmidb-1.0.0
print(DATA_DIR, ':', len([d for d in os.listdir(DATA_DIR) if d.startswith('S')]), 'dossiers de sujets')
if os.path.isdir(REPO_DIR):
    !git -C "$REPO_DIR" pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git log --oneline -1
!pip install -q mne
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU"
print('GPU :', torch.cuda.get_device_name(0))

## Outils : vérifier la référence et estimer les temps à partir des temps MESURÉS sur le GPU

In [ ]:
import json, glob
import numpy as np

def plis(dossier):
    return [json.load(open(f, encoding='utf-8')) for f in sorted(glob.glob(f'{OUT_DIR}/{dossier}/pli*_global.json'))]

def verifier_reference(n):
    ok = True
    for d in (f'cd_cnn_{n}cl', f'cnn_lstm32_{n}cl'):
        G = plis(d)
        print(f'{d:18s} : {len(G)}/5 plis', '' if len(G) == 5 else ' <-- référence incomplète')
        ok &= len(G) == 5
    return ok

def estimer_temps(n=4):
    """Estimation pour chaque variante (5 plis, sans adaptation), d'après les s/époque
    mesurés sur le GPU pendant la référence. La borne haute suppose 50 époques par pli."""
    t = {}
    for m, d in (('cd_cnn', f'cd_cnn_{n}cl'), ('cnn_lstm', f'cnn_lstm32_{n}cl')):
        G = plis(d)
        if not G:
            print(f'{d} : pas de mesure (référence absente)')
            return
        s = np.mean([np.mean(g['historique']['temps_s']) for g in G])
        ep = np.mean([g['epoques_faites'] for g in G])
        t[m] = (s * ep * 5 / 60, s * 50 * 5 / 60)
        print(f"{d:18s} : {s:.2f} s/époque sur {G[0]['device']}, {ep:.1f} époques/pli en moyenne")
    variantes = {'(a) --clip': ['cd_cnn', 'cnn_lstm'], '(b) cnn_lstm_moy': ['cnn_lstm'],
                 '(c) lr 5e-4': ['cd_cnn', 'cnn_lstm'], '(d) --norm-essai': ['cd_cnn', 'cnn_lstm']}
    total = [0, 0]
    for v, ms in variantes.items():
        a = sum(t[m][0] for m in ms); b = sum(t[m][1] for m in ms)
        total[0] += a; total[1] += b
        print(f'{v:18s} : environ {a:.0f} min (même nombre d\'époques que la référence), au plus {b:.0f} min')
    print(f'TOTAL 4 classes     : environ {total[0]:.0f} min, au plus {total[1]:.0f} min '
          f'(+ chargement des données, environ 5-10 min par session)')

verifier_reference(4)
estimer_temps(4)

## Variante (a) : `--clip` (norme du gradient ≤ 1,0), CD-CNN et CNN-LSTM-32, 4 classes
Dossiers : `cd_cnn_4cl_clip`, `cnn_lstm32_4cl_clip`.

In [ ]:
!python -W ignore train.py --model cd_cnn cnn_lstm --classes 4 --clip --sans-adaptation \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

## Variante (b) : `cnn_lstm_moy` (moyenne des 120 états cachés), CNN-LSTM seulement, 4 classes
C'est une nouvelle variante du CNN-LSTM : il n'y a pas d'équivalent pour le CD-CNN. Dossier : `cnn_lstm_moy32_4cl`.

In [ ]:
!python -W ignore train.py --model cnn_lstm --pool moyenne --classes 4 --sans-adaptation \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

## Variante (c) : `--lr 5e-4`, CD-CNN et CNN-LSTM-32, 4 classes
Dossiers : `cd_cnn_4cl_lr0.0005`, `cnn_lstm32_4cl_lr0.0005`.

In [ ]:
!python -W ignore train.py --model cd_cnn cnn_lstm --classes 4 --lr 5e-4 --sans-adaptation \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

## Variante (d) : `--norm-essai` (z-score de chaque fenêtre), CD-CNN et CNN-LSTM-32, 4 classes
Dossiers : `cd_cnn_4cl_normessai`, `cnn_lstm32_4cl_normessai`.

In [ ]:
!python -W ignore train.py --model cd_cnn cnn_lstm --classes 4 --norm-essai --sans-adaptation \
    --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

## Comparaison en 4 classes (lecture seule)
Le tableau est écrit dans `OUT_DIR/variantes_4cl.md`. **Envoie-moi ce fichier**, pour que je reporte toutes les variantes dans NOTES.md, y compris celles qui échouent.

In [ ]:
!python comparer_variantes.py --out-dir "$OUT_DIR" --classes 4

## Confirmation en 2 classes : à lancer SEULEMENT après décision

Indique la variante retenue **d'après la perte de validation** (tableau ci-dessus) :
- `'clip'`, `'moy'`, `'lr'` ou `'norm'` ;
- `None` si aucune variante ne bat la référence : il n'y a alors rien à confirmer.

Cette cellule :
1. ajoute l'adaptation au sujet (deux modes de z-score) aux plis 4 classes déjà entraînés de la variante, **sans réentraîner** ;
2. lance la variante en 2 classes, complète (entraînement et adaptation). Pour (a), (c) et (d), le CD-CNN est lancé avec la même option, pour l'équité.

In [ ]:
GAGNANTE = None   # 'clip' | 'moy' | 'lr' | 'norm'  (décidé sur la perte de validation)

OPTIONS = {'clip': '--model cd_cnn cnn_lstm --clip',
           'moy':  '--model cnn_lstm --pool moyenne',
           'lr':   '--model cd_cnn cnn_lstm --lr 5e-4',
           'norm': '--model cd_cnn cnn_lstm --norm-essai'}
assert GAGNANTE in OPTIONS, "Choisis d'abord la variante retenue (ou ne lance pas cette cellule)."
ARGS = OPTIONS[GAGNANTE]
!python -W ignore train.py $ARGS --classes 4 --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"
!python -W ignore train.py $ARGS --classes 2 --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"
!python comparer_variantes.py --out-dir "$OUT_DIR" --classes 2